# Blockchain Case Study — Risk Dashboard

> **Business question:** What can go wrong with our feature portfolio, and how much business value is at risk?

This notebook is your **risk dashboard**. Run it once and you get a complete picture of where value gets lost — and what to do about it.

---

### Who this notebook is for

| Role | What you get from this notebook |
|---|---|
| **Product Owner** | See which features survive risk, and which risk to reduce first |
| **Risk Manager** | Quantify portfolio-level risk by dimension, test sensitivity |
| **Engineering Lead** | Understand how development failure dominates total risk |

### How to read this notebook

Each section follows the same pattern:

1. **Question** — the business decision this section answers
2. **Output** — tables and charts generated from the simulation
3. **Reading the output** — plain-English guide explaining every number

> 💡 **Tip:** You don't need to understand the code. Focus on the coloured cards and tables — they contain the answers.

### Four risk dimensions in this model

| Risk | Name | What happens | How often |
|---|---|---|---|
| **Development** | Feature not completed | Business value → EUR 0 for that feature | Per feature (LLP rate) |
| **Market** | Market shock hits all features | Business value × 0.85 | 20% of scenarios |
| **Component** | Shared platform fails | Business value × 0.70 | Per dependency cluster |
| **Global** | Global crisis (regulation, pandemic) | Business value × 0.60 | 5% of scenarios |

**Quick navigation:** [Selected Portfolio](#2\)-Selected-Portfolio) · [Feature Profiles](#3\)-Feature-Risk-Profiles) · [Risk Waterfall](#4\)-Portfolio-Risk-Waterfall) · [Sensitivity](#5\)-Risk-Sensitivity)

---

### Key terms you will see

| Term | Plain English |
|---|---|
| **LLP** | Likelihood of non-completion — the chance a feature never reaches production |
| **Business Value Floor** | Your floor: 95 out of 100 scenarios produce a value *above* this number |
| **CVaR 95%** | Average of the worst 5% of scenarios — shows how bad the bad cases really are |
| **Retention rate** | How much of the original business value survives after all risks are applied |
| **Risk waterfall** | A step-by-step view: start with full value, subtract each risk layer |

---

*Previous: [04 — Portfolio Advisor](04-blockchain-case-study-advisor.ipynb) · Next: [06 — Development Risk & Profitability](06-blockchain-case-study-development-risk.ipynb)*

## 1) Setup — Load scenario and run simulation

The cell below loads the blockchain scenario from `config/blockchain.yaml` and initialises the risk simulation service. No changes needed — just run it.

In [ ]:
from fhs.application import AdvancedPortfolioService
from fhs.notebook import notebook_setup
from fhs.presentation.notebook import COLORS, show
from fhs.presentation.notebook.charts import (
    plot_delivery_market_resilience,
)

In [ ]:
setup = notebook_setup("blockchain")
scenario = setup.scenario
risk_model = setup.risk_model

if scenario is None or risk_model is None:
    raise RuntimeError("Scenario setup could not be initialized")

features = scenario.features
feature_names = [f.name for f in sorted(features, key=lambda x: x.name)]

In [ ]:
service = AdvancedPortfolioService(
    features,
    budget=scenario.budget,
    discount_rate=scenario.discount_rate,
    seed=scenario.seed,
    scenarios=scenario.scenarios,
)
_ = service.simulation_results

show.info(
    f"Source: <code>{scenario.config_path}</code><br>"
    f"Budget: <b>EUR {scenario.budget:,.0f}</b> · "
    f"Discount rate: <b>{scenario.discount_rate:.0%}</b><br>"
    f"Features: <b>{len(features)}</b> · "
    f"Scenarios: <b>{service.scenarios:,}</b>"
)

show.risk_model_configuration(risk_model)

---

## 2) Selected Portfolio (100% Budget)

### Question

**Which features did the optimizer select, and what is the starting point for risk analysis?**

The optimizer picked features using the `var_floor` strategy — it maximises the worst-case business value floor. This is the portfolio we test against all four risk dimensions below.

In [ ]:
ilp_result = service.optimize(
    solver="ilp", budget=scenario.budget, strategy="var_floor"
)
selected_names_list = list(ilp_result.recommended_features)
investment = float(ilp_result.total_cost)

show.optimizer(
    selected_names_list,
    investment,
    float(ilp_result.portfolio_expected),
    float(ilp_result.portfolio_var_95),
    float(scenario.budget) - investment,
    title="Portfolio Used for Risk Analysis (100% Budget)",
    objective="var_floor — maximise worst-case business value floor (downside protection)",
)
show.note(
    "All following risk sections analyse this portfolio. "
    "Individual feature profiles include all three features for comparison.",
    compact=True,
)

**Reading the output above**

- **Selected features** — the features the optimizer chose for the full budget.
- **Expected annual business value** — average outcome across all simulated scenarios. This is your planning number.
- **Business Value Floor** — in 95 out of 100 scenarios, your actual business value will be *above* this number. Think of it as the realistic worst case.

> 💡 All risk analysis below uses this exact portfolio. If you change the budget or strategy in Notebook 04, come back here to see updated risk profiles.

---

## Executive Risk Summary

> This is the bottom line. If you only read one thing, read this.


In [ ]:
# Executive risk summary — the answer first
portfolio_layers_summary = service.layers.simulate_portfolio_risk_layers(
    selected_names_list,
    risk_model=risk_model,
    seed=scenario.seed,
)
before_risk = portfolio_layers_summary.base.expected
after_risk = portfolio_layers_summary.after_risk_3.expected

# Dominant risk = largest loss dimension
delivery_loss_ = before_risk - portfolio_layers_summary.after_risk_1.expected
market_loss_ = (
    portfolio_layers_summary.after_risk_1.expected
    - portfolio_layers_summary.after_risk_2.expected
)
component_loss_ = (
    portfolio_layers_summary.after_risk_2.expected
    - portfolio_layers_summary.after_component.expected
)
global_loss_ = (
    portfolio_layers_summary.after_component.expected
    - portfolio_layers_summary.after_risk_3.expected
)
losses = {
    "Development failure": delivery_loss_,
    "Market shock": market_loss_,
    "Component failure": component_loss_,
    "Global crisis": global_loss_,
}
dominant = max(losses, key=losses.get)
dominant_pct = 100.0 * losses[dominant] / before_risk if before_risk > 0 else 0.0

In [ ]:
show.executive_risk_summary(
    before_risk,
    after_risk,
    investment,
    f"{dominant} ({dominant_pct:.0f}% of base value)",
)

**Reading the output above**

- **Business value at risk** — how much EUR the portfolio could lose due to all risk dimensions combined.
- **Net position after risk** — if this is negative (red), the expected business value after all risks is less than the investment. The portfolio is not expected to be profitable under the full risk model.
- **Dominant risk** — the single risk dimension that causes the most EUR damage.


---

## 3) Feature Risk Profiles — How much value does each feature keep?

Each feature passes through four risk gates in sequence. At each gate, some value is lost. The table below shows the expected business value remaining after each gate.

| Gate | What it does |
|---|---|
| **Base** | Full simulated business value — no risk applied yet |
| **After Development** | Features with high LLP lose most value here (if not completed → EUR 0) |
| **After Market** | Market shock reduces all surviving values |
| **After Component** | Platform dependency failure reduces value |
| **After Global** | Global crisis reduces remaining value |

> ⚠️ **The business value floor can be EUR 0 for features with high development risk.** When a feature's LLP exceeds 5%, more than 5% of scenarios produce zero business value (feature not completed). In that case, the floor is EUR 0 by design — not a bug. Use **expected business value** and **retention %** as your primary metrics for these features.

In [ ]:
feature_profiles = {}
for feature in sorted(features, key=lambda x: x.name):
    feature_profiles[feature.name] = service.layers.simulate_feature_risk_layers(
        feature.name, risk_model=risk_model, seed=scenario.seed
    )
risk_tables = service.layers.feature_risk_layer_tables(feature_profiles)
retention_matrix = service.layers.feature_risk_retention(feature_profiles)

show.feature_risk_layers(risk_tables)

In [ ]:
show.feature_risk_profile_cards(features, feature_profiles)
show.metrics(
    [
        (
            "Portfolio expected retention",
            f"{retention_matrix.portfolio_expected_retention_pct * 100:.1f}%",
            COLORS.warning,
        )
    ],
    title="Portfolio Retention at Full Risk",
)
show.feature_risk_decay_from_profiles(feature_names, feature_profiles)

**Reading the output above**

Look at the **retention rate** — the percentage of original business value that survives all risks:

- **High retention (>70%):** The feature is resilient. Risk layers don't erode much value.
- **Medium retention (40–70%):** Meaningful risk. Check which layer causes the biggest drop.
- **Low retention (<40%):** Most value is lost to risk. This feature needs de-risking before investment.

<div style="border: 2px solid currentColor; border-left: 6px solid #B06000; padding: 0.75em 1em; margin: 0.9em 0; background: transparent; color: inherit; border-radius: 4px; line-height: 1.55;">
<strong>Note — when business values are almost only positive:</strong> If a feature's simulated business value distribution shows little to no downside (gains dominate), retention will naturally be very high (&gt;90%). Do not over-interpret this as "risk-free". It usually reflects optimistic input assumptions (narrow spread, no negative tail). Cross-check <strong>Assumption Quality</strong> and the input ranges before treating high retention as a green light.
</div>

The **decay chart** shows business value shrinking from left to right through each risk layer. A steep drop at "After Development" means development failure is the dominant risk.

**What to do:**
- Product Owner: If retention is below 50%, consider reducing scope or development risk before committing budget.
- Risk Manager: Compare the EUR delta between layers — the biggest drop shows where mitigation has the most impact.

---

## 4) Portfolio Risk Waterfall — Where does value get lost?

### Question

**Across the whole portfolio, how much business value is lost at each risk layer — and which risk dimension causes the most damage?**

The waterfall traces the combined portfolio value through all four risk layers. Each step shows the expected business value *after* that risk has been applied.

In [ ]:
portfolio_layers = service.layers.simulate_portfolio_risk_layers(
    selected_names_list, risk_model=risk_model, seed=scenario.seed
)
show.portfolio_risk_waterfall_detail(portfolio_layers, risk_model, features, investment)

**Reading the output above**

**Risk waterfall table:** Read from top to bottom. The "Layer Loss" column shows how much value each risk dimension removes.

**Erosion chart (bar chart):** The tallest drop is the risk that matters most. Development failure is usually the largest because failed features produce zero value — wiping out a bigger share than market or crisis discounts.

**Profitability check:** The bottom card compares business value after all risks with the total investment.
- **Positive net profit** → the portfolio pays for itself even after all risks.
- **Negative net profit** → risk is eating more value than the investment generates. You need to reduce risk or cut features.

**Business Value Loss by Risk Dimension:** This table ranks the four risk dimensions by EUR impact. The top dimension is your #1 mitigation target.

> 💡 **Key insight:** Reducing development risk (LLP) typically has more impact than hedging against market or global crisis — because development failure sets business value to zero, while market/crisis only reduce it by a fraction.

## 4.1) Ranking-Strategien im Vergleich

### Question

**Welche Features sind je nach Entscheidungslogik vorne: absolute Sicherheit (`var_floor`) oder relative Effizienz (`risk_ratio`, `rorac`, `risk_adjusted_roi`)?**

Die vier Strategien sind **nicht äquivalent** und können zu unterschiedlichen Prioritäten führen:

- `var_floor`: absolute Business-Value-Untergrenze priorisieren (Vertrags-/Verpflichtungslogik)
- `risk_ratio`: geringsten relativen Drawdown bevorzugen (konservative Priorisierung)
- `rorac`: maximum return per unit of risk (standard risk-adjusted return metric)
- `risk_adjusted_roi`: besten Floor pro Investitions-Euro (CFO-Kapitalbudget-Sicht)

> Interpretation: Wenn sich die Top-Features zwischen Spalten unterscheiden, hängt die Entscheidung stark von der gewählten Risikoperspektive ab.

In [ ]:
strategies = ["var_floor", "risk_ratio", "rorac", "risk_adjusted_roi"]
strategy_labels = {
    "var_floor": "Floor protection",
    "risk_ratio": "Low erosion",
    "rorac": "Return per risk",
    "risk_adjusted_roi": "Capital efficiency",
}
strategy_colors = {
    "var_floor": COLORS.primary,
    "risk_ratio": COLORS.secondary,
    "rorac": COLORS.tertiary,
    "risk_adjusted_roi": COLORS.warning,
}
selected_set = set(selected_names_list)
ranking_by_strategy = {
    strategy: [
        row
        for row in service.decisions.rank_features(strategy=strategy)
        if row.feature in selected_set
    ]
    for strategy in strategies
}

feature_order = [f.name for f in features if f.name in selected_set]
feature_cards = []
for feature_name in feature_order:
    ranks = {}
    for strategy in strategies:
        ranking = ranking_by_strategy[strategy]
        ranks[strategy] = next(
            (
                idx
                for idx, row in enumerate(ranking, start=1)
                if row.feature == feature_name
            ),
            None,
        )

    available_ranks = [rank for rank in ranks.values() if rank is not None]
    avg_rank = sum(available_ranks) / len(available_ranks)
    best_rank = min(available_ranks)
    worst_rank = max(available_ranks)
    top_votes = sum(1 for rank in available_ranks if rank == 1)

    if best_rank == worst_rank:
        signal = "Full agreement"
        signal_border = COLORS.success
        action = "All strategies tell the same story."
    elif top_votes >= 2:
        signal = "Strong candidate"
        signal_border = COLORS.success
        action = "Multiple strategies put this feature at the top."
    elif worst_rank - best_rank >= 2:
        signal = "Strategy-dependent"
        signal_border = COLORS.warning
        action = "Good under one lens, weaker under another. Discuss the decision lens first."
    else:
        signal = "Stable middle"
        signal_border = COLORS.primary
        action = "Useful, but not the clearest first choice."

    badges = "".join(
        f'<span style="display:inline-block;border:1px solid {strategy_colors[strategy]};'
        f"border-left:4px solid {strategy_colors[strategy]};border-radius:4px;"
        f"padding:5px 8px;margin:3px 4px 3px 0;color:inherit;"
        f'background:transparent;font-size:12px;">'
        f"<b>{strategy_labels[strategy]}</b>: rank {ranks[strategy]}</span>"
        for strategy in strategies
    )

    feature_cards.append(
        f'<div style="border:1px solid {COLORS.border};border-left:5px solid {signal_border};'
        f"border-radius:6px;padding:14px 16px;background:transparent;color:inherit;"
        f'flex:1;min-width:260px;">'
        f'<div style="font-size:13px;text-transform:uppercase;letter-spacing:0.4px;'
        f'font-weight:bold;color:inherit;">{signal}</div>'
        f'<h4 style="margin:4px 0 8px 0;font-size:17px;color:inherit;">{feature_name}</h4>'
        f'<div style="font-size:13px;line-height:1.5;color:inherit;">'
        f"Average rank: <b>{avg_rank:.1f}</b> · Best/Worst: <b>{best_rank}/{worst_rank}</b> · Top votes: <b>{top_votes}</b>"
        f"</div>"
        f'<div style="margin-top:8px;">{badges}</div>'
        f'<div style="margin-top:10px;padding-top:8px;border-top:1px solid {COLORS.border};'
        f'font-size:13px;line-height:1.45;color:inherit;"><b>Decision read:</b> {action}</div>'
        f"</div>"
    )

html = (
    f'<div style="border:1px solid {COLORS.border};border-radius:6px;'
    f'padding:18px 22px;margin:12px 0;background:transparent;color:inherit;">'
    f'<h3 style="margin:0 0 8px 0;color:inherit;font-size:20px;">'
    f"Ranking Strategies — Decision Lens Comparison</h3>"
    f'<p style="margin:0 0 14px 0;color:inherit;font-size:14px;line-height:1.5;">'
    f"The old rank table showed positions, but not the decision meaning. "
    f"This view highlights whether strategies agree, split, or depend on the risk lens.</p>"
    f'<div style="display:flex;flex-wrap:wrap;gap:12px;align-items:stretch;">'
    f"{''.join(feature_cards)}</div>"
    f'<div style="margin-top:14px;padding:10px 14px;border-left:4px solid {COLORS.warning};'
    f'background:transparent;color:inherit;font-size:13px;line-height:1.55;">'
    f"<b>How to use it:</b> Pick the strategy that matches the governance question first. "
    f"Use <b>Floor protection</b> for downside commitments, <b>Low erosion</b> for resilience, "
    f"<b>Return per risk</b> for risk efficiency, and <b>Capital efficiency</b> for budget trade-offs."
    f"</div></div>"
)
show(html)

---

## 5) Risk Sensitivity — What if the world changes?

### Question

**How much would your portfolio value change if market or crisis risk turned out to be higher — or lower — than expected?**

Think of this as a **"what-if" check**: you keep the same features, same investment, but you ask *"what if the market gets worse?"* or *"what if we get lucky?"*

We test three scenarios for each risk factor:

| Scenario | Market shock chance | Global crisis chance | What it means |
|---|---|---|---|
| **Low Risk** (Optimistic) | 10% | 2.5% | Things go better than planned |
| **Base Case** (Expected) | 20% | 5.0% | Our central assumption |
| **High Risk** (Stress) | 30% | 7.5% | Things go worse — stress test |

> **What to look for:** If the High Risk bar is still above the orange investment line, the portfolio pays off even in the stress scenario. If it dips below — risk mitigation becomes urgent.

In [ ]:
# Development risk sensitivity (LLP factor: 0.5× = better testing, 1.5× = worse)
llp_factors = [0.5, 1.0, 1.5]
llp_results = {}
for factor in llp_factors:
    llp_results[factor] = service.layers.simulate_portfolio_risk_layers(
        selected_names_list,
        risk_model=risk_model,
        risk1_factor=factor,
        seed=scenario.seed,
    )

# Market risk sensitivity
r2_levels = [0.10, 0.20, 0.30]
r2_results = {}
for p2 in r2_levels:
    r2_results[p2] = service.layers.simulate_portfolio_risk_layers(
        selected_names_list,
        risk_model=risk_model,
        risk2_probability=p2,
        seed=scenario.seed,
    )

In [ ]:
# Global risk sensitivity
r3_levels = [0.025, 0.05, 0.075]
r3_results = {}
for p3 in r3_levels:
    r3_results[p3] = service.layers.simulate_portfolio_risk_layers(
        selected_names_list,
        risk_model=risk_model,
        risk3_probability=p3,
        seed=scenario.seed,
    )

# Component risk sensitivity
clusters = dict(risk_model.component_risk_by_cluster)
cluster_factors = [0.5, 1.0, 1.5]
comp_expected: dict[str, list[tuple[float, float]]] = {c: [] for c in sorted(clusters)}
for cluster_name, base_prob in sorted(clusters.items()):
    for factor in cluster_factors:
        test_prob = min(base_prob * factor, 0.95)
        result = service.layers.simulate_portfolio_risk_layers(
            selected_names_list,
            risk_model=risk_model,
            component_probability_by_cluster={cluster_name: test_prob},
            seed=scenario.seed,
        )
        comp_expected[cluster_name].append((test_prob, result.after_risk_3.expected))

In [ ]:
# Development × Market heatmap matrix (3×3)
heatmap_data = {}
for llp_f in llp_factors:
    for p2 in r2_levels:
        heatmap_data[(llp_f, p2)] = service.layers.simulate_portfolio_risk_layers(
            selected_names_list,
            risk_model=risk_model,
            risk1_factor=llp_f,
            risk2_probability=p2,
            seed=scenario.seed,
        ).after_risk_3.expected

#### How much is the portfolio worth — optimistic, normal, and stressed?

In [ ]:
# Simplified sensitivity — one combined table (market + global)
show.combined_sensitivity_table(
    r2_results,
    r3_results,
    r2_levels,
    r3_levels,
    title="Sensitivity — How Risk Changes Affect Portfolio Value",
)

#### Which risk combinations keep the portfolio profitable?

The heatmap below tests the two largest risk dimensions against each other:

- **Development Risk** (vertical axis) — the chance a feature is never delivered (LLP). We scale all feature failure rates uniformly: halved, unchanged, or increased by 50%.
- **Market Risk** (horizontal axis) — the chance of a market shock hitting all features at once (10%, 20%, or 30%).

Each cell shows the expected portfolio value after **all four risk layers** (including Component and Global at their base rates).

| Row | Scenario | What causes this |
|---|---|---|
| **Better development** (top) | All feature failure rates halved | Better testing, smaller scope, experienced team |
| **Base case** (middle) | Current failure rates unchanged | Your planning assumption |
| **Worse development** (bottom) | All feature failure rates +50% | New technology, team turnover, scope creep |

**How to read:** Green = profitable, Red = loss. The colour flips exactly at the break-even point (your investment). The KPI card below quantifies the EUR value of improving development.

In [ ]:
_, resilience = plot_delivery_market_resilience(
    heatmap_data,
    investment=investment,
    llp_factors=tuple(llp_factors),
    market_levels=tuple(r2_levels),
)

In [ ]:
_resilience_metrics = [
    (
        "Base case (current development, Market 20%)",
        f"EUR {resilience['base_k']:,.0f}k",
        COLORS.primary,
    ),
    (
        "If development risk halved",
        f"EUR {resilience['improved_k']:,.0f}k  (+{resilience['delivery_gain_k']:,.0f}k)",
        COLORS.success,
    ),
    (
        "If development risk worsens (+50%)",
        f"EUR {resilience['worsened_k']:,.0f}k  ({-resilience['delivery_loss_stress_k']:,.0f}k)",
        COLORS.danger,
    ),
    (
        "Value of development improvement",
        f"EUR {resilience['delivery_gain_k']:,.0f}k additional portfolio value",
        COLORS.success,
    ),
]

In [ ]:
show.metrics(
    _resilience_metrics,
    title="Development Risk Mitigation - Impact on Portfolio Value",
)

**Reading the heatmap**

The chart answers: *"If development risk and market risk change at the same time, does the portfolio stay profitable?"*

- **Green cells** — the portfolio comfortably covers the investment. Proceed with confidence.
- **Yellow/beige cells** — portfolio value is near break-even. The colour boundary marks exactly where profit turns to loss.
- **Red cells** — expected value falls below the investment. The portfolio loses money under these conditions.
- **White dashed border** — your base case (current assumptions).

**The key insight:** Moving *up* one row (improving development) typically gains more EUR than moving *left* one column (lower market risk). This confirms that **development risk mitigation has the highest return on effort** — because it is both the largest risk and the one within your control.

**What to do with this:**

| If you see... | It means... | Action |
|---|---|---|
| Top row all green | Investing in development quality makes the portfolio robust | Prioritise testing, team stability, smaller scope |
| Middle row mixed green/red | Base case is marginal — buffer is thin | Reduce development risk before committing full budget |
| Bottom row mostly red | If development worsens, the portfolio fails | Do not proceed without development de-risking |

> **Key takeaway:** Reducing development failure probability (through better testing, experienced teams, or smaller scope) is the single most impactful lever to protect portfolio profitability. Market risk is largely external — development risk is within the Product Owner's control.

---

## Excursus — How the Simulation Calculates Risk

> 📖 **You can skip this section.** It is here for transparency and audit purposes. All calculations above are fully automated. The actual parameter values are shown in the **Risk Model Configuration** card at the top of this notebook.

This is the only formula reference in this notebook. All calculations use the configured Monte Carlo scenario count.

### Development Risk — Development gate (per feature)

Each scenario draws a random number. If it falls below the Likelihood of non-completion (LLP), the feature is not completed and produces zero business value:

$$\text{Business Value after Development} = \begin{cases} \text{Simulated Business Value} & \text{if completed} \\ 0 & \text{if not completed} \end{cases}$$

$$P(\text{not completed}) = \text{LLP (per feature, from config)}$$

### Market Risk — Market shock (portfolio-wide)

One random draw per scenario. If a market shock occurs, all feature values are multiplied by the market factor:

$$\text{Business Value after Market} = \text{Business Value after Development} \times \begin{cases} m_{\text{market}} & \text{if market shock} \\ 1.0 & \text{otherwise} \end{cases}$$

$$P(\text{market shock}) = p_{\text{market}}, \quad m_{\text{market}} \text{ — both from risk model config}$$

### Component Risk (per cluster)

Features sharing a dependency cluster experience the same failure draw:

$$\text{Business Value after Component} = \text{Business Value after Market} \times \begin{cases} m_{\text{comp}} & \text{if platform outage} \\ 1.0 & \text{otherwise} \end{cases}$$

$$P(\text{outage}), \; m_{\text{comp}} \text{ — per cluster, from risk model config}$$

### Global Risk — Global crisis (portfolio-wide)

One random draw per scenario. If a global crisis occurs, all remaining values are reduced:

$$\text{Business Value after Global} = \text{Business Value after Component} \times \begin{cases} m_{\text{global}} & \text{if global crisis} \\ 1.0 & \text{otherwise} \end{cases}$$

$$P(\text{global crisis}) = p_{\text{global}}, \quad m_{\text{global}} \text{ — both from risk model config}$$

### Business Value Floor and CVaR 95%

$$\text{Business Value Floor} = \text{5th percentile of } N \text{ business value scenarios}$$

$$\text{CVaR 95\%} = \text{average of scenarios} \leq \text{Business Value Floor}$$

---

## Decision Brief — What to do next

### For the Product Owner

1. **Check feature retention** (Section 3): If any feature retains less than 50% of its base value, de-risk development before investing further.
2. **Find the biggest risk** (Section 4): The "Business Value Loss by Risk Dimension" table tells you which risk to tackle first — usually development failure.
3. **Set risk tolerance** (Section 5): Use the sensitivity range to decide how much risk swing is acceptable for your budget.

### For the Risk Manager

1. **Quantify total risk exposure** (Section 4): Compare "Total expected risk loss" with the portfolio investment. If loss approaches investment, raise a flag.
2. **Stress-test assumptions** (Section 5): The high-risk column shows business value under stressed probabilities. Use this for board reporting.
3. **Identify concentration risk** (Section 3): Features with low retention and high cost create disproportionate risk.

### For the board

| Question | Answer (from this notebook) |
|---|---|
| Does the portfolio pay off after risks? | Section 4 → Profitability Check card |
| What is the single biggest risk? | Section 4 → Business Value Loss table (top row) |
| How bad can it get? | Section 5 → Sensitivity high column |
| What should we mitigate first? | Section 4 → Largest loss dimension = first mitigation target |

In [ ]:
# Dynamic key takeaway — derived from simulation results
_llp_values = [f.likelihood_of_non_delivery for f in features]
_min_llp = min(_llp_values)
_max_llp = max(_llp_values)

_sorted_losses = sorted(losses.items(), key=lambda x: x[1], reverse=True)
_second_risk, _second_loss = (
    _sorted_losses[1] if len(_sorted_losses) > 1 else (None, 0.0)
)
_second_pct = (
    100.0 * _second_loss / before_risk if _second_risk and before_risk > 0 else 0.0
)

_llp_range = (
    f"{_min_llp:.0%}–{_max_llp:.0%}" if _min_llp != _max_llp else f"{_min_llp:.0%}"
)

_msg = (
    f"The biggest risk for this portfolio is <b>{dominant}</b> — "
    f"accounting for <b>{dominant_pct:.0f}%</b> of base value erosion "
    f"(LLP range: {_llp_range} across features). "
)
if _second_risk:
    _msg += f"<b>{_second_risk}</b> is the next largest driver at {_second_pct:.0f}%. "
_msg += "Any model assuming full development completion will significantly overestimate returns."

show.note(f"⚠️ <b>Key takeaway:</b> {_msg}")

---

## Industry Benchmarks — How Do Your Numbers Compare?

The metrics in this notebook have equivalents in financial industry practice. Use these reference points to judge whether your portfolio risk is typical or unusual:

| Metric | Benchmark | Source / Convention |
|---|---|---|
| **HHI (concentration)** | HHI > 0.25 = highly concentrated | EU/US competition authorities use this threshold for market dominance |
| **IRR vs. hurdle** | IRR > WACC + 3% = typically investable | Standard corporate finance buffer for estimation uncertainty |
| **Risk Ratio** | < 0.3 low · 0.3–0.5 moderate · > 0.5 high | Coefficient of variation thresholds used in project risk management |
| **CVaR / floor spread** | CVaR < 5× floor = well-behaved tail | Common risk calibration norm (also used in Swiss/EU banking governance) |
| **Retention rate** | > 70% resilient · 40–70% moderate · < 40% fragile | Derived from stress-testing practice in operational risk |

> **These benchmarks are reference points, not rules.** Your organization may have different thresholds depending on industry, risk appetite, and regulatory context. A tech startup and a regulated utility will judge the same HHI very differently.

---

## Notebook Navigation

| # | Notebook | What you learn |
|:-:|----------|----------------|
| 01 | [Getting Started](01-getting-started.ipynb) | One feature, simulation basics, business value floor dashboard |
| 02 | [Blockchain Case Study](02-blockchain-case-study.ipynb) | Feature risks, portfolio baseline, board recommendation |
| 03 | [Capital Budgeting](03-blockchain-case-study-capital-budgeting.ipynb) | NPV, IRR, upfront vs. installment financing |
| 04 | [Portfolio Advisor](04-blockchain-case-study-advisor.ipynb) | ILP optimisation, budget sensitivity |
| **05** | **Risk Dashboard** | **← You are here** |
| 06 | [Development Risk & Profitability](06-blockchain-case-study-development-risk.ipynb) | Sprint overruns, cost simulation, break-even |
| 07 | [Executive Decision](07-blockchain-case-study-decision.ipynb) | All dimensions in one view, combined recommendation |
| A01 | [Portfolio Advisor](advanced/01-portfolio-advisor.ipynb) | Solver comparison, feature ranking, HHI |
| A02 | [Portfolio Risk Dashboard](advanced/02-portfolio-risk-dashboard.ipynb) | Risk layers, stress scenarios, budget risk path |

---

### Quick Reference

| Term | Definition |
|------|------------|
| **LLP** | Likelihood of non-completion — probability the feature fails to reach production. At LLP 20%, 1 in 5 paths produces zero business value. |
| **Risk Waterfall** | Sequential risk layers: Development → Market → Component → Global. Each layer reduces business value further. |
| **Retention Rate** | Fraction of base business value that survives all risk layers. Low retention = high aggregate risk. |
| **Business Value Floor 95** | 5th percentile of the simulation. 95% of scenarios exceed this floor. |
| **CVaR 95%** | Average of the worst 5%. More conservative than the floor alone. |
| **Market Risk** | Portfolio-wide shock: all features lose a fraction of value simultaneously. |
| **Global Risk** | Severe crisis (regulation, pandemic) — low probability, high impact across the portfolio. |
| **Component Risk** | Cluster-level risk: features sharing a dependency fail together. |
| **Sensitivity** | Testing how results change when you increase or decrease a risk parameter. |

*Previous: [04 — Portfolio Advisor](04-blockchain-case-study-advisor.ipynb) · Next: [06 — Development Risk & Profitability](06-blockchain-case-study-development-risk.ipynb)*

---
**Previous:** [NB 04: Portfolio Advisor](04-blockchain-case-study-advisor.ipynb) | **Next:** [NB 06: Development Risk](06-blockchain-case-study-development-risk.ipynb)
